In [1]:
import rospy
rospy.init_node('analyzer_node', anonymous=True)

In [3]:
from PyKDL import Frame, Rotation, Vector
from robot_command.interfaces.joints_to_pose_interface import (
    JointsToPoseInterface,
)

joints_to_pose = JointsToPoseInterface()

def calc_tcp_distance2(times, points):
    poses_in = [
        joints_to_pose.get_pose_from_joint_values(point[:6])
        for point in points
    ]
    frames = [
        Frame(
            Rotation.Quaternion(
                p.orientation.x, p.orientation.y, p.orientation.z, p.orientation.w
            ),
            Vector(p.position.x, p.position.y, p.position.z),
        )
        for p in poses_in
    ]
    tcp_pos = 0.0
    tcp_rot = 0.0
    tcp_lin = [0.0]
    tcp_rot = [0.0]
    #rot_interp = RotationalInterpolationSingleAxis()
    for i in range(1, len(frames)):
        diff = frames[i] * frames[i-1].Inverse()
        tcp_pos += diff.p.Norm()
        tcp_lin.append(tcp_pos)
    tcp_vel = [0.0]
    tcp_vel.extend((tcp_lin[i] - tcp_lin[i-1]) / (times[i] - times[i-1]) for i in range(1, len(tcp_lin)))
    tcp_accel = [0.0]
    tcp_accel.extend((tcp_vel[i] - tcp_vel[i-1]) / ((times[i] - times[i-1]) + (times[i-1] - times[max(0, i-2)])) * 2.0 for i in range(1, len(tcp_vel)))
    return tcp_lin, tcp_vel, tcp_accel


def calc_tcp_distance(trajectory):
    times = [point.time_from_start.to_sec() for point in trajectory.points]
    return times, calc_tcp_distance2(times, [point.positions for point in trajectory.points])